# House Price Predictor EDA

We are asking a simple question: how can we use a small set of house details to estimate a price?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

path = Path('..') / 'data' / 'houses.csv'
df = pd.read_csv(path)
df.head()

### 1) Load the data

The first step is just to read the dataset and take a quick look. We want to be sure the columns make sense before using them in a model.

In [ ]:
df.shape

In [ ]:
df.columns.tolist()

### 2) Check for missing values

If a column has missing values, we need to decide whether to fill them or remove them before training.

In [ ]:
df.isna().sum()

In [ ]:
df.describe().round(2)

### 3) Explore the price distribution

This chart helps us see where most houses sit in price and whether there are any unusual values.

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df['house_price'], bins=20, kde=True, color='#E07856')
plt.title('House Price Distribution')
plt.xlabel('House price')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

### 4) Look for basic relationships

A house price usually rises with area, so a scatter plot is a helpful place to start.

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='area_sqft', y='house_price', alpha=0.7, color='#7BA88F')
plt.title('House Area vs. Price')
plt.xlabel('Area (sq ft)')
plt.ylabel('Price')
plt.tight_layout()
plt.show()

### 5) Create a couple of useful features

We are not trying to overcomplicate this. We want just a few simple ideas that make the raw information easier for the model to use.

In [ ]:
df['area_per_bedroom'] = df['area_sqft'] / df['bedrooms'].replace(0, 1)
df['house_age_group'] = pd.cut(
    df['age_years'],
    bins=[-1, 5, 20, float('inf')],
    labels=['new', 'mid_age', 'older'],
    right=False,
)
df[['area_per_bedroom', 'house_age_group']].head()

### 6) Prepare for training

We split the data so some houses are used to learn and some are hidden until the end for evaluation.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

feature_columns = [
    'area_sqft', 'bedrooms', 'bathrooms', 'age_years', 'location_score', 'parking_spaces', 'area_per_bedroom'
]
X = df[feature_columns]
y = df['house_price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LinearRegression()
model.fit(X_train, y_train)
preds = model.predict(X_test)

mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)
print(f'MAE: ${mae:,.0f}')
print(f'RMSE: ${rmse:,.0f}')
print(f'R2: {r2:.3f}')

### 7) Try a sample prediction

This lets us see the model act on a new house profile, even though it is still a training project.

In [ ]:
sample_house = {
    'area_sqft': 2100,
    'bedrooms': 3,
    'bathrooms': 2.5,
    'age_years': 12,
    'location_score': 76,
    'parking_spaces': 2
}
row = pd.DataFrame([sample_house])
row['area_per_bedroom'] = row['area_sqft'] / row['bedrooms'].replace(0, 1)
model.predict(row[feature_columns])[0]